In [1]:
import json

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0813 12:01:16.533000 12432 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
data = load_dataset("MathLLMs/MathCodeInstruct", split="train")

In [3]:
print(data)

Dataset({
    features: ['messages'],
    num_rows: 79067
})


In [4]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

==((====))==  Unsloth 2026.8.15: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.


In [5]:
def format_content(contents: list[dict[str, str]]):
    strings = []
    for item in contents:
        t = item["type"]
        c = item["content"]
        if t == "text":
            strings.append(c.strip())
        elif t == "code":
            strings.append(f"```python\n"
                           f"{c.strip()}\n"
                           f"```")
        elif t == "execution":
            strings.append(f"```output\n"
                           f"{c.strip()}\n"
                           f"```")
        else:
            raise ValueError(f"unhandled block type: {t}")
    return "\n\n".join(strings)

def format_example(example):
    formatted_messages = [
        {
            "role": msg["role"],
            "content": format_content(msg["content"])
        }
        for msg in example["messages"]
    ]


    return {"text": tokenizer.apply_chat_template(formatted_messages, tokenize=False)}

In [6]:
ds = data.map(format_example, remove_columns="messages")

Map:   0%|          | 0/79067 [00:00<?, ? examples/s]

In [9]:
def filter(element):
    return len(tokenizer(element["text"])["input_ids"]) < max_seq_length

In [10]:
ds = ds.filter(filter).shuffle(seed=123)

Filter:   0%|          | 0/79067 [00:00<?, ? examples/s]

In [12]:
print(ds)

Dataset({
    features: ['text'],
    num_rows: 78201
})


In [13]:
lens = []
for i in range(ds.num_rows):
    lens.append(len(tokenizer(ds[i]["text"])["input_ids"]))
print(max(lens))

2046


In [14]:
eval_ds    = ds.select(range(len(ds) - 1000, len(ds)))   # last 1000 rows
train_pool = ds.select(range(len(ds) - 1000))

with open("mathcodeinstruct-eval.json", "w", encoding="utf-8") as f:
    json.dump(eval_ds.to_list(), f, ensure_ascii=False)

for k in [5, 10, 20]:
    with open(f"mathcodeinstruct-train-{k}k.json", "w", encoding="utf-8") as f:
        json.dump(train_pool.select(range(k * 1000)).to_list(), f, ensure_ascii=False)